In [ ]:
import json
from pathlib import Path

import numpy
import pyccl
from matplotlib import pyplot

In [ ]:
base_path = Path.cwd().parent
code_path = base_path / 'Code'
data_path = base_path / 'Data'
figure_path = base_path / 'Figure'
print(base_path, code_path, data_path, figure_path)

In [ ]:
import numpy
import pyccl
from matplotlib import pyplot

## 5. A fiducial cosmology

We initially keep cosmology fixed so that we can understand the IA parameters one at a
time. The Eisenstein–Hu transfer function avoids requiring a separate CAMB installation.
PyCCL uses distances in Mpc and wavenumbers in $\mathrm{Mpc}^{-1}$.

In [ ]:
with open(data_path / 'Planck.json', 'r') as file:
    parameter = json.load(file)
print(parameter)

In [ ]:
cosmology = pyccl.cosmology.Cosmology(
    h = parameter['H'],
    w0 = parameter['W0'],
    wa = parameter['WA'],
    A_s = parameter['AS'], 
    n_s = parameter['NS'], 
    m_nu = parameter['MNU'], 
    T_CMB = parameter['TCMB'],
    Omega_k = parameter['OMEGAK'], 
    Omega_c = parameter['OMEGAC'], 
    Omega_b = parameter['OMEGAB'], 
    mass_split='normal', transfer_function = 'boltzmann_camb', 
    extra_parameters = {'camb': {'kmax': 100, 'lmax': 5000, 'halofit_version': 'mead2020_feedback', 'HMCode_logT_AGN': 7.8}}
)

In [ ]:
z1 = 0.0
z2 = 3.0
z_size = 10
z_grid = numpy.linspace(z1, z2, z_size)

## 9. Assemble the complete IA response

We now combine:

$$
\frac{1}{D(z)}
\times
R_z(z;\eta)
\times
R_L(z;\xi,z_q,s)
\times
S(k;q,k_t,n).
$$

Explicitly,

$$
\boxed{
R_zR_L
=
\left(\frac{1+z}{1+z_p}\right)^\eta
\left[
\frac{
1+\left(\dfrac{1+z}{1+z_q}\right)^s
}{
1+\left(\dfrac{1+z_p}{1+z_q}\right)^s
}
\right]^{\xi/s}.
}
$$

The function below returns both the complete two-dimensional response $F(z,k)$ and
its separate factors. Keeping the factors available makes the model easier to debug
and explain.

In [ ]:
def ia_response(
    cosmo,
    z,
    k,
    A0=1.0,
    eta=0.0,
    xi=0.0,
    z_q=1.0,
    luminosity_sharpness=2.0,
    q=0.0,
    k_transition=0.2,
    scale_sharpness=2.0,
):
    z = numpy.asarray(z)
    k = numpy.asarray(k)

    a = 1.0 / (1.0 + z)
    growth = pyccl.growth_factor(cosmo, a)
    R_z = redshift_factor(z, eta)
    R_L = luminosity_factor(
        z,
        xi=xi,
        z_q=z_q,
        s=luminosity_sharpness,
    )
    S_k = scale_transition(k, q, k_transition, scale_sharpness)

    redshift_amplitude = (
        -A0
        * C0
        * cosmo["Omega_m"]
        / growth
        * R_z
        * R_L
    )

    F = redshift_amplitude[:, None] * S_k[None, :]
    factors = {
        "growth": growth,
        "R_z": R_z,
        "R_L": R_L,
        "S_k": S_k,
        "redshift_amplitude": redshift_amplitude,
    }
    return F, factors

## 10. Generate fiducial 3D IA spectra

We evaluate the nonlinear matter spectrum on the same $(z,k)$ grid and then apply the
IA response. The resulting arrays contain one curve per redshift.

In [ ]:
k = numpy.logspace(-3, 0.5, 160)
z = numpy.asarray([0.2, 0.5, 0.8, 1.2, 1.6, 2.0])

P_m = numpy.vstack([
    pyccl.nonlin_matter_power(
        cosmology,
        k,
        1.0 / (1.0 + z_value),
    )
    for z_value in z
])

fiducial_parameters = {
    "A0": 1.0,
    "eta": 0.0,
    "xi": 1.0,
    "z_q": 1.0,
    "luminosity_sharpness": 3.0,
    "q": 1.0,
    "k_transition": 0.2,
    "scale_sharpness": 2.0,
}

F, factors = ia_response(
    cosmology,
    z,
    k,
    **fiducial_parameters,
)

P_deltaI = F * P_m
P_II = F**2 * P_m

print("P_m shape:      ", P_m.shape)
print("F shape:        ", F.shape)
print("P_deltaI shape: ", P_deltaI.shape)
print("P_II shape:     ", P_II.shape)

### 10.1 Inspect the separate redshift factors

$1/D(z)$, $R_z$, and $R_L$ can all change the amplitude with redshift, but they
represent different assumptions. This also means their parameters may be partly
degenerate in a future inference analysis.

In [ ]:
fig, axes = pyplot.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(z, 1.0 / factors["growth"], marker="o")
axes[0].set_title("Standard growth scaling")
axes[0].set_ylabel(r"$1/D(z)$")

axes[1].plot(z, factors["R_z"], marker="o")
axes[1].set_title("Extra redshift factor")
axes[1].set_ylabel(r"$R_z(z;\eta)$")

axes[2].plot(z, factors["R_L"], marker="o")
axes[2].set_title("Effective luminosity factor")
axes[2].set_ylabel(r"$R_L(z;\xi,z_q,s)$")
axes[2].axvline(
    fiducial_parameters["z_q"],
    color="tab:red",
    linestyle="--",
    label=r"$z_q$",
)
axes[2].legend()

for ax in axes:
    ax.set_xlabel("redshift z")
    ax.axvline(Z_PIVOT, color="grey", linestyle=":")

fig.tight_layout()
pyplot.show()

### 10.2 Plot the spectra

For positive $A_0$, our convention gives $P_{\delta I}<0$. A logarithmic axis
cannot display negative numbers, so the middle panel shows $-P_{\delta I}$ and labels
this explicitly. We never silently discard the sign in the saved data.

In [ ]:
fig, axes = pyplot.subplots(1, 3, figsize=(17, 5))

for index, z_value in enumerate(z):
    label = rf"$z={z_value:.1f}$"
    axes[0].plot(k, P_m[index], label=label)
    axes[1].plot(k, -P_deltaI[index], label=label)
    axes[2].plot(k, P_II[index], label=label)

titles = [
    "Nonlinear matter spectrum",
    "Matter–intrinsic spectrum",
    "Intrinsic–intrinsic spectrum",
]
ylabels = [
    r"$P_{\rm m}(k,z)$ [Mpc$^3$]",
    r"$-P_{\delta I}(k,z)$ [Mpc$^3$]",
    r"$P_{II}(k,z)$ [Mpc$^3$]",
]

for ax, title, ylabel in zip(axes, titles, ylabels):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.tight_layout()
pyplot.show()